### Utility Functions

In [1]:
# This part helps with stability on multi-GPU systems with great inbalance between the GPUs (eg. integrated vs discrete gpu)
# IMPORTANT must be done before importing torch, else session must be restarted
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0]
    total = torch.cuda.mem_get_info(i)[1]

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free:  {free / 1024**3:.2f} GB")
    print(f"  Total: {total / 1024**3:.2f} GB")

GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU
  Free:  5.00 GB
  Total: 6.00 GB


In [ ]:
from util import clear_folder

# Clears results of the last run, not necessary, just to reduce folder size

clear_folder("./results")

In [3]:
from util import clear_cuda_cache

# If model gets stuck during training, uncomment the following line

clear_cuda_cache()

c:\Users\dkraj\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Training Pipeline

## Load Dataset

This portion loads dataset, and assigns id for each label

#### English

This loads english version of the dataset

In [5]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-combined",
    split="train",
    label2id=label2id
)


Loaded datasets (train=44323, val=5540, test=5541)


#### Slovenian

This loads slovenian version of the dataset

In [4]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id
)


Loaded datasets (train=44323, val=5540, test=5541)


## Model settings 

### XLM-RoBERTA

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "FacebookAI/xlm-roberta-base"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### TinyBert

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "huawei-noah/TinyBERT_General_4L_312D"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### mBERT

In [5]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "bert-base-multilingual-cased"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8608.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

## Train

In [17]:
from datetime import datetime
from util import clear_folder

clear_folder("./results")

trainer.train()

model_save_folder = "trained_models" 

model_save_name = model_name.split("/")[-1] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
trainer.save_model(os.path.join(model_save_folder, model_save_name))
tokenizer.save_pretrained(os.path.join(model_save_folder, model_save_name))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.123732,0.200549,0.963899,0.955209,0.947826,0.951503
2,0.112166,0.171142,0.969856,0.958554,0.960870,0.959710
3,0.039785,0.210840,0.970758,0.959981,0.961836,0.960907


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

('trained_models\\bert-base-multilingual-cased2026-05-07_22-06-34\\tokenizer_config.json',
 'trained_models\\bert-base-multilingual-cased2026-05-07_22-06-34\\tokenizer.json')

In [7]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.678501,0,0.628881,0.602941,0.019807,0.038354


{'eval_loss': 0.6785014271736145,
 'eval_accuracy': 0.6288808664259928,
 'eval_precision': 0.6029411764705882,
 'eval_recall': 0.019806763285024155,
 'eval_f1': 0.03835360149672591}

### Data Visualization

In [8]:
import graphs
import importlib

importlib.reload(graphs)

graphs.generate_all_plots("./results", "./graphs" + "/" + model_name.split("/")[-1])

# Testing pipeline

In [8]:
from util import load_latest_trained_model_and_tokenizer, build_trainer

model, tokenizer, latest_dir = load_latest_trained_model_and_tokenizer(
    trained_models_root="./trained_models",
    num_labels=2,
 )
print("Loaded:", latest_dir)


trainer = build_trainer(model, train_dataset, val_dataset)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3703.00it/s]


Loaded: ./trained_models\bert-base-multilingual-cased2026-05-07_22-06-34


## Load a previously trained model (skip training)

If you already trained a model once, you can reload it from `./trained_models` and run evaluation/zero-shot testing without training again.

# Zero-shot testing (EN -> SL)

Evaluate the model trained on the English dataset on the Slovenian dataset without any further training.

In [6]:
from util import load_datasets_from_hf, tokenize_dataset


sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id,
    train_size=0.8,
    val_size=0.1,
    seed=42,
 )

sl_test_ds_tok = tokenize_dataset(sl_test_ds, tokenizer)

trainer.eval_dataset = sl_test_ds_tok

zero_shot_metrics = trainer.evaluate()


Loaded datasets (train=44323, val=5540, test=5541)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.738037,0,0.367262,0.367016,0.967492,0.532159


{'eval_loss': 0.738036572933197,
 'eval_accuracy': 0.36726222703483125,
 'eval_precision': 0.3670163813730904,
 'eval_recall': 0.9674915089762252,
 'eval_f1': 0.5321590605817987}

In [9]:
from util import predict_labels
import graphs

labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

preds, labels = predict_labels(trainer, sl_test_ds_tok)
graph_dir = os.path.join("./graphs", os.path.basename(latest_dir))
graphs.plot_confusion_matrix(
    y_true=labels,
    y_pred=preds,
    labels=labels_in_order,
    save_path=os.path.join(graph_dir, "zero_shot_en_to_sl_confusion_matrix.png"),
)
print(f"Saved confusion matrix to {os.path.join(graph_dir, 'zero_shot_en_to_sl_confusion_matrix.png')}")

Saved confusion matrix to ./graphs\bert-base-multilingual-cased2026-05-07_22-06-34\zero_shot_en_to_sl_confusion_matrix.png
